In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')


# ------------------------------------------------------------
# 1. Channel-agnostic feature aggregator (same as before)
# ------------------------------------------------------------
def build_channel_agnostic_features(df, band_suffixes=None):
    if band_suffixes is None:
        band_suffixes = [
            '_delta_power', '_theta_power', '_alpha_power',
            '_beta_power', '_gamma_power',
            '_delta_relative', '_theta_relative', '_alpha_relative',
            '_beta_relative', '_gamma_relative'
        ]

    agg = pd.DataFrame(index=df.index)

    for suf in band_suffixes:
        cols = [c for c in df.columns if c.endswith(suf)]
        if cols:
            vals = df[cols].values
            agg[f'{suf[1:]}_mean'] = vals.mean(axis=1)
            agg[f'{suf[1:]}_std'] = vals.std(axis=1)

    entropy_cols = [c for c in df.columns if c.endswith('_entropy')]
    if entropy_cols:
        vals = df[entropy_cols].values
        agg['entropy_mean'] = vals.mean(axis=1)
        agg['entropy_std'] = vals.std(axis=1)

    return agg


# ------------------------------------------------------------
# 2. CORAL domain adaptation
# ------------------------------------------------------------
def coral_transform(Xs, Xt):
    cov_s = np.cov(Xs, rowvar=False) + 1e-6 * np.eye(Xs.shape[1])
    cov_t = np.cov(Xt, rowvar=False) + 1e-6 * np.eye(Xt.shape[1])

    Es, Ls, _ = np.linalg.svd(cov_s)
    Et, Lt, _ = np.linalg.svd(cov_t)

    Cs_inv_sqrt = Es @ np.diag(1/np.sqrt(Ls)) @ Es.T
    Ct_sqrt = Et @ np.diag(np.sqrt(Lt)) @ Et.T

    A = Cs_inv_sqrt @ Ct_sqrt
    return Xs @ A


# ------------------------------------------------------------
# 3. Subject-wise splitting helper
# ------------------------------------------------------------
def subject_wise_split(df, subject_col='subject_id', test_size=0.5):
    subjects = df[subject_col].unique()
    rng = np.random.default_rng(seed=42)
    rng.shuffle(subjects)

    split_idx = int(len(subjects) * (1 - test_size))
    train_subjects = subjects[:split_idx]
    test_subjects  = subjects[split_idx:]

    train_df = df[df[subject_col].isin(train_subjects)].copy()
    test_df  = df[df[subject_col].isin(test_subjects)].copy()

    print("\n=== SUBJECT-WISE SPLIT ===")
    print(f" Train subjects ({len(train_subjects)}): {train_subjects}")
    print(f" Test subjects  ({len(test_subjects)}): {test_subjects}")
    print(f" Train samples: {train_df.shape[0]}")
    print(f" Test samples:  {test_df.shape[0]}")

    return train_df, test_df


# ------------------------------------------------------------
# 4. Load EEGMAT & SAM40 and align feature spaces
# ------------------------------------------------------------
def load_and_align_datasets(eegmat_file='eeg_features.csv',
                            sam40_file='sam40_eeg_features.csv'):

    # ---- EEGMAT (source) ----
    eegmat = pd.read_csv(eegmat_file)
    eegmat = eegmat[eegmat['task_type'].isin(['REST', 'MATH'])].copy()
    eegmat['task_type'] = eegmat['task_type'].replace({'REST': 'RELAX'})

    print("\nEEGMAT source distribution:")
    print(eegmat['task_type'].value_counts())

    # ---- SAM40 (target) ----
    sam40 = pd.read_csv(sam40_file)
    sam40 = sam40[sam40['task_type'].isin(['RELAX', 'MATH'])].copy()

    print("\nSAM40 target distribution:")
    print(sam40['task_type'].value_counts())

    # Remove metadata columns
    meta_cols = ['file_name','task_type','subject_id','duration','num_channels','sampling_rate']

    X_eegmat_raw = eegmat.drop(columns=[c for c in meta_cols if c in eegmat.columns])
    y_eegmat = eegmat['task_type']
    sub_eegmat = eegmat['subject_id']

    X_sam40_raw = sam40.drop(columns=[c for c in meta_cols if c in sam40.columns])
    y_sam40 = sam40['task_type']
    sub_sam40 = sam40['subject_id']

    # Channel-agnostic feature set
    X_eegmat_agg = build_channel_agnostic_features(X_eegmat_raw)
    X_sam40_agg  = build_channel_agnostic_features(X_sam40_raw)

    # Intersection of feature names
    common = sorted(set(X_eegmat_agg.columns) & set(X_sam40_agg.columns))
    X_eegmat_agg = X_eegmat_agg[common].copy()
    X_sam40_agg  = X_sam40_agg[common].copy()

    print(f"\nCommon features: {len(common)}\n")

    # Add subject IDs for splitting
    X_eegmat_agg['subject_id'] = sub_eegmat.values
    X_sam40_agg['subject_id']  = sub_sam40.values

    # Add label
    X_eegmat_agg['label'] = y_eegmat.values
    X_sam40_agg['label'] = y_sam40.values

    return X_eegmat_agg, X_sam40_agg, common


In [2]:
# ------------------------------------------------------------
# 5. Full domain adaptation experiment with SUBJECT SPLITS
# ------------------------------------------------------------
def run_cross_domain_subject_split():
    X_src, X_tgt, features = load_and_align_datasets()

    # -------------------------
    # SAM40 subject-wise split
    # -------------------------
    sam40_train, sam40_test = subject_wise_split(X_tgt, 'subject_id', test_size=0.5)

    # Build X/y consistently
    y_src = X_src['label'].map({'RELAX':0, 'MATH':1}).values
    y_tgt_train = sam40_train['label'].map({'RELAX':0, 'MATH':1}).values
    y_tgt_test  = sam40_test['label'].map({'RELAX':0, 'MATH':1}).values

    # Extract feature matrices
    cols = features
    X_src_feat = X_src[cols].values
    X_tgt_train_feat = sam40_train[cols].values
    X_tgt_test_feat  = sam40_test[cols].values

    # -------------------------
    # Standardize (using source statistics)
    # -------------------------
    scaler = StandardScaler()
    X_src_scaled = scaler.fit_transform(X_src_feat)
    X_tgt_train_scaled = scaler.transform(X_tgt_train_feat)
    X_tgt_test_scaled  = scaler.transform(X_tgt_test_feat)

    # -------------------------
    # BASELINE: Train EEGMAT → Test SAM40 (subject split)
    # -------------------------
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_src_scaled, y_src)
    pred_base = clf.predict(X_tgt_test_scaled)

    print("\n=== BASELINE (no domain adaptation) ===")
    print("Train: EEGMAT | Test: SAM40 (subject-held-out)")
    print("Accuracy:", accuracy_score(y_tgt_test, pred_base))
    print("F1:", f1_score(y_tgt_test, pred_base))
    print(classification_report(y_tgt_test, pred_base, target_names=['RELAX','MATH']))

    # -------------------------
    # CORAL domain adaptation
    # -------------------------
    X_src_coral = coral_transform(X_src_scaled, X_tgt_train_scaled)

    clf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf2.fit(X_src_coral, y_src)
    pred_coral = clf2.predict(X_tgt_test_scaled)

    print("\n=== CORAL DOMAIN ADAPTATION ===")
    print("Train: CORAL(EEGMAT→SAM40) | Test: SAM40 subj-held-out")
    print("Accuracy:", accuracy_score(y_tgt_test, pred_coral))
    print("F1:", f1_score(y_tgt_test, pred_coral))
    print(classification_report(y_tgt_test, pred_coral, target_names=['RELAX','MATH']))

    # -------------------------
    # In-domain SAM40 subject-split upper bound
    # -------------------------
    clf3 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf3.fit(X_tgt_train_scaled, y_tgt_train)
    pred_in = clf3.predict(X_tgt_test_scaled)

    print("\n=== SAM40 IN-DOMAIN UPPER BOUND ===")
    print("Train: SAM40 subjects | Test: SAM40 held-out subjects")
    print("Accuracy:", accuracy_score(y_tgt_test, pred_in))
    print("F1:", f1_score(y_tgt_test, pred_in))
    print(classification_report(y_tgt_test, pred_in, target_names=['RELAX','MATH']))

    return {
        "baseline_acc": accuracy_score(y_tgt_test, pred_base),
        "baseline_f1": f1_score(y_tgt_test, pred_base),
        "coral_acc": accuracy_score(y_tgt_test, pred_coral),
        "coral_f1": f1_score(y_tgt_test, pred_coral),
        "in_domain_acc": accuracy_score(y_tgt_test, pred_in),
        "in_domain_f1": f1_score(y_tgt_test, pred_in)
    }


if __name__ == "__main__":
    results = run_cross_domain_subject_split()
    print("\nSummary:", results)



EEGMAT source distribution:
task_type
RELAX    72
MATH     72
Name: count, dtype: int64

SAM40 target distribution:
task_type
MATH     120
RELAX    120
Name: count, dtype: int64

Common features: 22


=== SUBJECT-WISE SPLIT ===
 Train subjects (20): [ 3 17  9 27 14  7 25 15 33 36  2 29  5 32  1 35 39 38 34 16]
 Test subjects  (20): [37 20 31 19  6 30 28 13 10 24 26  4 21 23  8 12 40 11 22 18]
 Train samples: 120
 Test samples:  120

=== BASELINE (no domain adaptation) ===
Train: EEGMAT | Test: SAM40 (subject-held-out)
Accuracy: 0.5333333333333333
F1: 0.28205128205128205
              precision    recall  f1-score   support

       RELAX       0.52      0.88      0.65        60
        MATH       0.61      0.18      0.28        60

    accuracy                           0.53       120
   macro avg       0.57      0.53      0.47       120
weighted avg       0.57      0.53      0.47       120


=== CORAL DOMAIN ADAPTATION ===
Train: CORAL(EEGMAT→SAM40) | Test: SAM40 subj-held-out
Accurac